# Data Analysis - PBI Dataset
Analisis data penjualan menggunakan BigQuery

## Setup & Autentikasi

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import pandas as pd

PROJECT_ID = 'da-2025-481513'
client = bigquery.Client(project=PROJECT_ID)

# Helper function untuk query BigQuery
def execute(query):
    return client.query(query).to_dataframe()

# Helper function untuk membuat chart Top N (mudah diganti tipe chart-nya)
def plot_top_chart(
    df,
    label_col,
    value_col,
    title='',
    xlabel='',
    ylabel='',
    chart_type='bar',   # Ganti ke 'pie' untuk pie chart
    figsize=(10, 5),
    palette='Blues_r'
):
    """Plot Top-N chart. Ubah chart_type='bar' atau 'pie'."""
    fig, ax = plt.subplots(figsize=figsize)

    if chart_type == 'bar':
        colors = sns.color_palette(palette, len(df))
        bars = ax.barh(df[label_col], df[value_col], color=colors)
        ax.invert_yaxis()
        ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
        ax.set_xlabel(ylabel)
        ax.set_ylabel(xlabel)
        for bar in bars:
            width = bar.get_width()
            ax.text(
                width * 1.01, bar.get_y() + bar.get_height() / 2,
                f'{width:,.2f}', va='center', fontsize=9
            )
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    elif chart_type == 'pie':
        colors = sns.color_palette(palette, len(df))
        wedges, texts, autotexts = ax.pie(
            df[value_col],
            labels=df[label_col],
            autopct='%1.1f%%',
            colors=colors,
            startangle=140
        )
        ax.set_title(title, fontsize=14, fontweight='bold', pad=12)

    else:
        raise ValueError("chart_type harus 'bar' atau 'pie'")

    plt.tight_layout()
    plt.show()

# Helper function untuk menampilkan metric card
def show_metric(label, value, prefix='', suffix=''):
    print(f"{'='*40}")
    print(f"  {label}")
    print(f"  {prefix}{value:,.2f}{suffix}")
    print(f"{'='*40}")

print('Setup selesai ✓')

---
## Bab 1 - Data Cleaning

### 1.1 Cek Duplikat

In [ ]:
duplikat = execute("""
SELECT 'product_data'         AS tabel, COUNT(*) - COUNT(DISTINCT ProdNumber)  AS jumlah_duplikat FROM `da-2025-481513.pbi.product_data`
    UNION ALL
    SELECT 'customers_data'       AS tabel, COUNT(*) - COUNT(DISTINCT CustomerID)  AS jumlah_duplikat FROM `da-2025-481513.pbi.customers_data`
    UNION ALL
    SELECT 'orders_data'          AS tabel, COUNT(*) - COUNT(DISTINCT OrderID)     AS jumlah_duplikat FROM `da-2025-481513.pbi.orders_data`
    UNION ALL
    SELECT 'productCategory_data' AS tabel, COUNT(*) - COUNT(DISTINCT CategoryID)  AS jumlah_duplikat FROM `da-2025-481513.pbi.productCategory_data`;
""")
duplikat

In [ ]:
# Visualisasi jumlah duplikat per tabel
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#d9534f' if v > 0 else '#5cb85c' for v in duplikat['jumlah_duplikat']]
bars = ax.bar(duplikat['tabel'], duplikat['jumlah_duplikat'], color=colors)
ax.set_title('Jumlah Duplikat per Tabel', fontsize=13, fontweight='bold')
ax.set_xlabel('Tabel')
ax.set_ylabel('Jumlah Duplikat')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(int(bar.get_height())), ha='center', fontsize=10)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 1.2 Cek Data Null

In [ ]:
null_check = execute("""
SELECT 'product_data'  AS tabel,
    COUNTIF(ProdNumber IS NULL) AS null_ProdNumber,
    COUNTIF(ProdName   IS NULL) AS null_ProdName,
    COUNTIF(Price      IS NULL) AS null_Price
FROM `da-2025-481513.pbi.product_data`
  UNION ALL
  SELECT 'customers_data' AS tabel,
    COUNTIF(CustomerID    IS NULL) AS null_CustomerID,
    COUNTIF(CustomerEmail IS NULL) AS null_CustomerEmail,
    COUNTIF(CustomerCity  IS NULL) AS null_CustomerCity
FROM `da-2025-481513.pbi.customers_data`;
""")
null_check

### 1.3 Cek Penulisan Email

In [ ]:
email_invalid = execute("""
SELECT CustomerID, CustomerEmail
  FROM `da-2025-481513.pbi.customers_data`
  WHERE CustomerEmail LIKE '%#%'
  LIMIT 5;
""")
email_invalid

### 1.4 Cek Harga (Price)

In [ ]:
price_check = execute("""
SELECT
  MIN(Price)        AS harga_minimum,
  MAX(Price)        AS harga_maksimum,
  MIN(Price / 100)  AS harga_minimum_fix,
  MAX(Price / 100)  AS harga_maksimum_fix
FROM `da-2025-481513.pbi.product_data`;
""")
price_check

In [ ]:
# Metric: Rentang harga setelah fix
row = price_check.iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
metrics = [
    ('Harga Minimum (fix)', row['harga_minimum_fix']),
    ('Harga Maksimum (fix)', row['harga_maksimum_fix']),
]
for ax, (label, val) in zip(axes, metrics):
    ax.text(0.5, 0.55, f'Rp {val:,.2f}', ha='center', va='center',
            fontsize=18, fontweight='bold', color='#2c7bb6')
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=10, color='#555')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_facecolor('#f0f4fa')
    for spine in ax.spines.values(): spine.set_visible(False)
plt.suptitle('Rentang Harga Produk (setelah /100)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Bab 2 - Menentukan Primary Key (Soal Nomor 1)

In [ ]:
product_data = execute("select * from `da-2025-481513.pbi.product_data`")
product_data.head()

In [ ]:
customers_data = execute("select * from `da-2025-481513.pbi.customers_data`")
customers_data.head()

In [ ]:
orders_data = execute("select * from `da-2025-481513.pbi.orders_data`")
orders_data.head()

In [ ]:
category_data = execute("select * from `da-2025-481513.pbi.productCategory_data`")
category_data.head()

In [ ]:
# Catatan: Di BigQuery primary key ditetapkan via DDL (NOT ENFORCED)
# ALTER TABLE pbi.product_data ADD PRIMARY KEY (ProdNumber) NOT ENFORCED;
# ALTER TABLE pbi.customers_data ADD PRIMARY KEY (CustomerID) NOT ENFORCED;
# ALTER TABLE pbi.orders_data ADD PRIMARY KEY (OrderID) NOT ENFORCED;
# ALTER TABLE pbi.productCategory_data ADD PRIMARY KEY (CategoryID) NOT ENFORCED;
print('Primary Key Summary:')
pk_info = [
    ('product_data', 'ProdNumber'),
    ('customers_data', 'CustomerID'),
    ('orders_data', 'OrderID'),
    ('productCategory_data', 'CategoryID'),
]
for tabel, pk in pk_info:
    print(f'  {tabel:<25} → PK: {pk}')

---
## Bab 3 - Transaksi Pertama per Kategori per Tanggal (Soal Nomor 3)

In [ ]:
first_transactions = execute("""
CREATE OR REPLACE TABLE `da-2025-481513.pbi.master_table` AS
WITH ranked AS (
  SELECT
    o.Date                       AS order_date,
    pc.CategoryID,
    pc.CategoryName              AS category_name,
    p.ProdName                   AS product_name,
    p.Price / 100                AS product_price,
    o.Quantity                   AS order_qty,
    (o.Quantity * p.Price / 100) AS total_sales,
    c.CustomerEmail              AS cust_email,
    c.CustomerCity               AS cust_city,
    ROW_NUMBER() OVER (
      PARTITION BY o.Date, pc.CategoryID
      ORDER BY o.OrderID ASC
    ) AS rn
  FROM
    `da-2025-481513.pbi.orders_data` o
  JOIN
    `da-2025-481513.pbi.customers_data` c
    ON o.CustomerID = c.CustomerID
  JOIN
    `da-2025-481513.pbi.product_data` p
    ON o.ProdNumber = p.ProdNumber
  JOIN
    `da-2025-481513.pbi.productCategory_data` pc
    ON p.Category = pc.CategoryID
)

SELECT
  order_date,
  category_name,
  product_name,
  product_price,
  order_qty,
  total_sales,
  cust_email,
  cust_city
FROM ranked
WHERE rn = 1
ORDER BY
  order_date ASC,
  CategoryID ASC;
""")
first_transactions

---
## Bab 4 - Analisis Penjualan (Soal Nomor 4)

### 4.1 Total Sales Keseluruhan

In [ ]:
total_sales_df = execute("""
SELECT
  ROUND(SUM(o.Quantity * p.Price / 100), 2) AS total_sales
    FROM `da-2025-481513.pbi.orders_data` o
    JOIN `da-2025-481513.pbi.product_data` p
      ON o.ProdNumber = p.ProdNumber
WHERE o.Quantity > 0 AND p.Price > 0;
""")

total_sales_value = total_sales_df['total_sales'].iloc[0]

# Metric card
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.text(0.5, 0.60, f'Rp {total_sales_value:,.2f}', ha='center', va='center',
        fontsize=22, fontweight='bold', color='#2c7bb6')
ax.text(0.5, 0.25, 'Total Sales Keseluruhan', ha='center', va='center',
        fontsize=11, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_facecolor('#eef4fb')
plt.tight_layout()
plt.show()

### 4.2 Total Sales & Qty by Category Product

In [ ]:
sales_by_category = execute("""
SELECT
  pc.CategoryName AS category_name,
  ROUND(SUM(o.Quantity * p.Price / 100), 2) AS total_sales
FROM `da-2025-481513.pbi.orders_data` o
JOIN `da-2025-481513.pbi.product_data` p ON o.ProdNumber = p.ProdNumber
JOIN `da-2025-481513.pbi.productCategory_data` pc ON p.Category = pc.CategoryID
WHERE o.Quantity > 0 AND p.Price > 0
GROUP BY category_name
ORDER BY total_sales DESC;
""")
sales_by_category

In [ ]:
# ========================
# Ganti chart_type='pie' untuk pie chart
# ========================
plot_top_chart(
    df=sales_by_category,
    label_col='category_name',
    value_col='total_sales',
    title='Total Sales by Category Product',
    xlabel='Kategori',
    ylabel='Total Sales (Rp)',
    chart_type='bar',   # Ubah ke 'pie' untuk pie chart
    palette='Blues_r'
)

In [ ]:
qty_by_category = execute("""
SELECT
  pc.CategoryName AS category_name,
  SUM(o.Quantity) AS total_qty
FROM `da-2025-481513.pbi.orders_data` o
JOIN `da-2025-481513.pbi.product_data` p ON o.ProdNumber = p.ProdNumber
JOIN `da-2025-481513.pbi.productCategory_data` pc ON p.Category = pc.CategoryID
WHERE o.Quantity > 0
GROUP BY category_name
ORDER BY total_qty DESC;
""")
qty_by_category

In [ ]:
# ========================
# Ganti chart_type='pie' untuk pie chart
# ========================
plot_top_chart(
    df=qty_by_category,
    label_col='category_name',
    value_col='total_qty',
    title='Total Quantity by Category Product',
    xlabel='Kategori',
    ylabel='Total Qty',
    chart_type='bar',   # Ubah ke 'pie' untuk pie chart
    palette='Greens_r'
)

### 4.3 Total Sales & Qty by City

In [ ]:
sales_by_city = execute("""
SELECT
  INITCAP(TRIM(c.CustomerCity)) AS cust_city,
  ROUND(SUM(o.Quantity * p.Price / 100), 2) AS total_sales
FROM `da-2025-481513.pbi.orders_data` o
JOIN `da-2025-481513.pbi.product_data` p ON o.ProdNumber = p.ProdNumber
JOIN `da-2025-481513.pbi.customers_data` c ON o.CustomerID = c.CustomerID
WHERE o.Quantity > 0 AND p.Price > 0 AND c.CustomerCity IS NOT NULL
GROUP BY cust_city
ORDER BY total_sales DESC;
""")
sales_by_city

In [ ]:
# ========================
# Ganti chart_type='pie' untuk pie chart
# ========================
plot_top_chart(
    df=sales_by_city,
    label_col='cust_city',
    value_col='total_sales',
    title='Total Sales by City',
    xlabel='Kota',
    ylabel='Total Sales (Rp)',
    chart_type='bar',   # Ubah ke 'pie' untuk pie chart
    palette='Purples_r'
)

In [ ]:
qty_by_city = execute("""
SELECT
  INITCAP(TRIM(c.CustomerCity)) AS cust_city,
  SUM(o.Quantity) AS total_qty
FROM `da-2025-481513.pbi.orders_data` o
JOIN `da-2025-481513.pbi.customers_data` c ON o.CustomerID = c.CustomerID
WHERE o.Quantity > 0 AND c.CustomerCity IS NOT NULL
GROUP BY cust_city
ORDER BY total_qty DESC;
""")
qty_by_city

In [ ]:
# ========================
# Ganti chart_type='pie' untuk pie chart
# ========================
plot_top_chart(
    df=qty_by_city,
    label_col='cust_city',
    value_col='total_qty',
    title='Total Quantity by City',
    xlabel='Kota',
    ylabel='Total Qty',
    chart_type='bar',   # Ubah ke 'pie' untuk pie chart
    palette='Oranges_r'
)

### 4.4 Top 5 Category Product by Sales

In [ ]:
top5_sales = execute("""
SELECT
  pc.CategoryName AS category_name,
  ROUND(SUM(o.Quantity * p.Price / 100), 2) AS total_sales
FROM `da-2025-481513.pbi.orders_data` o
JOIN `da-2025-481513.pbi.product_data` p ON o.ProdNumber = p.ProdNumber
JOIN `da-2025-481513.pbi.productCategory_data` pc ON p.Category = pc.CategoryID
WHERE o.Quantity > 0 AND p.Price > 0
GROUP BY category_name
ORDER BY total_sales DESC
LIMIT 5;
""")
top5_sales

In [ ]:
# ========================
# Ganti chart_type='pie' untuk pie chart
# ========================
plot_top_chart(
    df=top5_sales,
    label_col='category_name',
    value_col='total_sales',
    title='Top 5 Category Product by Sales',
    xlabel='Kategori',
    ylabel='Total Sales (Rp)',
    chart_type='bar',   # Ubah ke 'pie' untuk pie chart
    palette='YlOrRd'
)

### 4.5 Top 5 Category Product by Qty

In [ ]:
top5_qty = execute("""
SELECT
  pc.CategoryName AS category_name,
  SUM(o.Quantity) AS total_qty
FROM `da-2025-481513.pbi.orders_data` o
JOIN `da-2025-481513.pbi.product_data` p ON o.ProdNumber = p.ProdNumber
JOIN `da-2025-481513.pbi.productCategory_data` pc ON p.Category = pc.CategoryID
WHERE o.Quantity > 0
GROUP BY category_name
ORDER BY total_qty DESC
LIMIT 5;
""")
top5_qty

In [ ]:
# ========================
# Ganti chart_type='pie' untuk pie chart
# ========================
plot_top_chart(
    df=top5_qty,
    label_col='category_name',
    value_col='total_qty',
    title='Top 5 Category Product by Qty',
    xlabel='Kategori',
    ylabel='Total Qty',
    chart_type='bar',   # Ubah ke 'pie' untuk pie chart
    palette='cool'
)